# Lab 14: Parameter-Efficient Fine-Tuning (PEFT) with LoRA & QLoRA

Welcome to Laboratory 14! In this lab, we master **Parameter-Efficient Fine-Tuning (PEFT)** using **Low-Rank Adaptation (LoRA)** (*Hu et al., 2021*):
1. **The Fine-Tuning Bottleneck**: Why full parameter fine-tuning of 70B+ parameter models is computationally prohibitive.
2. **Low-Rank Matrix Factorization**: Implement $\mathbf{W} = \mathbf{W}_0 + \frac{\alpha}{r} \mathbf{B}\mathbf{A}$ from scratch in PyTorch.
3. **Parameter Efficiency Analysis**: Quantify $>99\%$ reductions in trainable parameter counts while preserving model capacity.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import PyTorch neural network modules
import torch
import torch.nn as nn

# Seed for reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Low-Rank Adaptation (LoRA) Layer from First Principles

### Architecture & Mathematical Formulation: `LoRALinear`
For a pretrained linear projection layer with frozen weights $\mathbf{W}_0 \in \mathbb{R}^{d_{out} \times d_{in}}$:
* **Weight Update Decomposition**: Instead of updating all $d_{out} \times d_{in}$ parameters, LoRA factorizes the weight update $\Delta \mathbf{W}$ into two low-rank matrices:
  $$\mathbf{h} = \mathbf{W}_0 \mathbf{x} + \Delta \mathbf{W} \mathbf{x} = \mathbf{W}_0 \mathbf{x} + \frac{\alpha}{r} \mathbf{B} \mathbf{A} \mathbf{x}$$
  where:
  * $\mathbf{A} \in \mathbb{R}^{r \times d_{in}}$ is initialized with Gaussian noise $\mathcal{N}(0, \sigma^2)$.
  * $\mathbf{B} \in \mathbb{R}^{d_{out} \times r}$ is initialized to zero, ensuring $\Delta \mathbf{W} = 0$ at the start of training.
  * $r \ll \min(d_{in}, d_{out})$ is the adaptation rank (e.g. $r=4$ or $8$).
  * $\alpha$ is a constant scaling hyperparameter.


In [ ]:
# Define the Custom Low-Rank Adaptation (LoRA) Linear Layer
class LoRALinear(nn.Module):
    """Applies Low-Rank Adaptation to a Linear layer with frozen base weights."""
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 8.0):
        super(LoRALinear, self).__init__()
        self.rank = rank
        self.scaling = alpha / rank # Constant scaling factor alpha / r
        
        # 1. Base Pretrained Linear Layer (Weights are frozen!)
        self.base_layer = nn.Linear(in_features, out_features, bias=False)
        self.base_layer.weight.requires_grad = False # Freeze base parameters
        
        # 2. Trainable Low-Rank Adaptation Matrices A and B
        # Matrix A: projects down from in_features -> rank (initialized with Gaussian noise)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        
        # Matrix B: projects up from rank -> out_features (initialized to zero)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Standard frozen forward pass: x @ W_0^T
        base_output = self.base_layer(x)
        
        # Low-rank adapter forward pass: (x @ A^T @ B^T) * (alpha / r)
        # Tensor shapes: x (B, in_features) -> (B, rank) -> (B, out_features)
        lora_output = (x @ self.lora_A.t()) @ self.lora_B.t() * self.scaling
        
        # Sum frozen and adapter activations
        return base_output + lora_output
    
    def merge_weights(self):
        """Permanently fuses LoRA weights into base weights for zero inference latency."""
        with torch.no_grad():
            self.base_layer.weight.data += (self.lora_B @ self.lora_A) * self.scaling

# Instantiate LoRALinear layer (Input: 1024 -> Output: 1024, Rank r=8, Alpha=16)
d_in, d_out, rank = 1024, 1024, 8
lora_layer = LoRALinear(in_features=d_in, out_features=d_out, rank=rank, alpha=16.0).to(device)

# Quantify Parameter Reduction
frozen_params = d_in * d_out
trainable_lora_params = (d_in * rank) + (rank * d_out)
param_reduction_pct = (1.0 - trainable_lora_params / frozen_params) * 100.0

print(f'Base Model Frozen Parameters:    {frozen_params:,}')
print(f'Trainable LoRA Parameters (r={rank}): {trainable_lora_params:,}')
print(f'Parameter Storage Reduction:     {param_reduction_pct:.2f}% fewer trainable parameters!')

# Test forward pass with dummy batch
dummy_input = torch.randn(4, 1024).to(device)
output_tensor = lora_layer(dummy_input)
print(f'\nOutput Tensor Shape: {output_tensor.shape}')


## 3. Summary & Key Takeaways
1. **Parameter Efficiency**: LoRA achieves identical task performance while updating $<1\%$ of total network parameters.
2. **Zero Inference Latency**: Adapters can be folded directly into base weights $\mathbf{W} = \mathbf{W}_0 + \frac{\alpha}{r} \mathbf{B}\mathbf{A}$ at deployment time.
3. **Modular Storage**: Multiple task-specific adapters (each only a few megabytes) can share a single frozen base model in production.
